In [2]:
# ============================================
# Add Predictions to Stock Data
# Author: Samaneh Kavianfar
# ============================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sqlalchemy import create_engine, URL

# ============================================
# 1. Connect to PostgreSQL
# ============================================
url = URL.create(
    drivername='postgresql',
    username='postgres',
    password='@Taha1391',
    host='localhost',
    port=5432,
    database='stock_db'
)

engine = create_engine(url)
# ============================================
# 2. Load data
# ============================================
df = pd.read_sql("SELECT * FROM fct_daily_returns ORDER BY date", engine)

# Create lag features
df['close_lag1'] = df['close'].shift(1)
df['close_lag7'] = df['close'].shift(7)
df = df.dropna()

# ============================================
# 3. Train model
# ============================================
X = df[['close_lag1', 'close_lag7']]
y = df['close']

split_idx = int(len(df) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ============================================
# 4. Make predictions
# ============================================
df['predicted_close'] = model.predict(X)

# ============================================
# 5. Save results to PostgreSQL
# ============================================
df[['date', 'symbol', 'predicted_close']].to_sql(
    'stock_predictions', engine, if_exists='replace', index=False
)

print("✅ Predictions added to 'stock_predictions' table!")

✅ Predictions added to 'stock_predictions' table!
